# 02 — DistilBERT: Official Baseline

**Project:** Explainability-Guided Feature Analysis for Reducing False Positives in Phishing Email Detection Without Increasing False Negatives  
**Experiment:** Official DistilBERT baseline, prior to explainability-guided interventions

This notebook presents the **recorded official baseline configuration and evaluation** from the original DistilBERT experiment. It retains the project's established TP/TN terminology. It does **not** train or reload the model, run inference, or claim to reproduce the original training run.

## 1. Experimental purpose

The official baseline establishes the reference performance against which later DistilBERT interventions were evaluated. The research objective was to **reduce false positives without increasing false negatives**.

The original experiment used **label 0 for legitimate email** and **label 1 for phishing email**. The following table follows the terminology already used in this project's original experimental records.

| **Project term** | **Meaning** |
|:--|:--|
| **TP** | Legitimate email correctly classified as legitimate |
| **FP** | Legitimate email incorrectly classified as phishing |
| **FN** | Phishing email incorrectly classified as legitimate |
| **TN** | Phishing email correctly classified as phishing |

**Project convention:** TP means correctly classified legitimate emails, and TN means correctly classified phishing emails throughout this project. All tables retain these original names and recorded values. Precision, recall, and F1-score are calculated for phishing (label 1).

## 2. Recorded dataset and split

The experiment used the Kaggle **Phishing Email Dataset (Nazario_5 and TREC07)**, file `email_text.csv`.

| **Dataset measure** | **Number of emails** |
|:--|--:|
| Original records | 53,668 |
| Records removed during cleaning | 91 |
| Records after cleaning | **53,577** |
| Training records | 42,861 |
| Test records | **10,716** |

The recorded split was **80% training / 20% testing**, stratified by class, with `random_state=42`.

| **Email class** | **Training** | **Test** | **Cleaned total** |
|:--|--:|--:|--:|
| Legitimate (0) | 18,991 | 4,748 | 23,739 |
| Phishing (1) | 23,870 | 5,968 | 29,838 |
| **Total** | **42,861** | **10,716** | **53,577** |

The original text preparation was documented in Notebook 01. No new preprocessing is performed here.

## 3. Recorded model and training configuration

| **Setting** | **Original experiment** |
|:--|:--|
| Pretrained checkpoint | `distilbert-base-uncased` |
| Task | Binary email classification |
| Maximum token length | 256 |
| Batch size | 8 |
| Optimizer | AdamW |
| Learning rate | `2e-5` |
| Training epochs | 2 |
| Random seed | 42 |
| Recorded accelerator | Google Colab Tesla T4 GPU |
| Official saved model location | `/content/drive/MyDrive/DistilBERT_Official_Baseline` |

These settings describe the **recorded** experiment. This notebook does not substitute a newly written training implementation for the original source code.

## 4. Official baseline classification outcomes

The following values were reported in the original official baseline evaluation and are retained **without modification**.

| **Project classification outcome** | **Count** | **Interpretation** |
|:--|--:|:--|
| TP | 4,717 | Correct legitimate |
| FP | **31** | Legitimate classified as phishing |
| FN | **22** | Phishing classified as legitimate |
| TN | 5,946 | Correct phishing |
| **Total** | **10,716** | All test emails |

**Confusion matrix** (rows show actual classes; columns show predicted classes):

| **Actual class** | **Predicted legitimate (0)** | **Predicted phishing (1)** | **Total** |
|:--|--:|--:|--:|
| Legitimate (0) | **4,717** | 31 | **4,748** |
| Phishing (1) | 22 | **5,946** | **5,968** |
| **Total** | **4,739** | **5,977** | **10,716** |

The same recorded matrix is shown below in array form:

```text
[[4717,   31],
 [  22, 5946]]
```

The project labels above are maintained as recorded; matrix positions follow the class order **[legitimate, phishing]**.

## 5. Official baseline performance

| **Performance measure** | **Recorded value** |
|:--|--:|
| Accuracy | **0.995054** |
| Precision (phishing) | **0.994813** |
| Recall (phishing) | **0.996314** |
| F1-score (phishing) | **0.995563** |
| ROC-AUC | **0.999262** |
| False positives | **31** |
| False negatives | **22** |

The classification metrics use **phishing (label 1)** as the target class, as in the original evaluation. The TP/TN names in the preceding table follow the project convention. The six-decimal metrics are the recorded results; they are not invented or estimated from an unexecuted model.

## 6. Arithmetic verification of the recorded results

The following cell checks that the published counts and classification metrics are internally consistent. This is **verification code**, not the original model-training or inference code. It does not load the dataset or the checkpoint.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import numpy as np

# Recorded official baseline counts (project terminology).
project_tp_correct_legitimate = 4717
project_fp = 31
project_fn = 22
project_tn_correct_phishing = 5946

# Matrix order: rows = actual [legitimate, phishing]; columns = predicted [legitimate, phishing].
recorded_cm = np.array([
    [project_tp_correct_legitimate, project_fp],
    [project_fn, project_tn_correct_phishing],
], dtype=int)

assert recorded_cm.shape == (2, 2)
assert recorded_cm.sum() == 10_716
assert recorded_cm[0].sum() == 4_748
assert recorded_cm[1].sum() == 5_968

# Reconstruct labels from aggregate counts for arithmetic checking only.
y_true = np.repeat([0, 1], [4_748, 5_968])
y_pred = np.concatenate([
    np.repeat(0, project_tp_correct_legitimate),
    np.repeat(1, project_fp),
    np.repeat(0, project_fn),
    np.repeat(1, project_tn_correct_phishing),
])

assert np.array_equal(confusion_matrix(y_true, y_pred, labels=[0, 1]), recorded_cm)

checks = {
    "Accuracy": (accuracy_score(y_true, y_pred), 0.995054),
    "Precision": (precision_score(y_true, y_pred, pos_label=1), 0.994813),
    "Recall": (recall_score(y_true, y_pred, pos_label=1), 0.996314),
    "F1-score": (f1_score(y_true, y_pred, pos_label=1), 0.995563),
}
for name, (calculated, recorded) in checks.items():
    assert abs(calculated - recorded) < 1e-6, f"{name} differs from the recorded result"

print("OFFICIAL DISTILBERT BASELINE — RECORDED RESULTS VERIFIED")
print("Confusion matrix (legitimate, phishing):")
print(recorded_cm)
print(f"Test records: {recorded_cm.sum():,}")
print(f"FP: {project_fp} | FN: {project_fn}")
for name, (calculated, _) in checks.items():
    print(f"{name}: {calculated:.6f}")
print("Recorded ROC-AUC: 0.999262 (not recomputed from aggregate counts)")

# Present the same verified metrics in a neatly formatted table.
import pandas as pd
from IPython.display import display

verification_table = pd.DataFrame(
    [
        {"Performance measure": name, "Recorded": f"{recorded:.6f}",
         "Recalculated": f"{calculated:.6f}", "Verification": "PASS"}
        for name, (calculated, recorded) in checks.items()
    ]
)
verification_table.loc[len(verification_table)] = {
    "Performance measure": "ROC-AUC",
    "Recorded": "0.999262",
    "Recalculated": "Not available from aggregate counts",
    "Verification": "Not recalculated",
}

display(
    verification_table.style
    .hide(axis="index")
    .set_caption("Official DistilBERT Baseline — Numerical Verification")
    .set_properties(**{"padding": "7px 12px", "border": "1px solid #cbd5e1", "font-size": "12px"})
    .set_properties(subset=["Recorded", "Recalculated", "Verification"],
                    **{"text-align": "center"})
    .set_table_styles([
        {"selector": "caption", "props": [("caption-side", "top"), ("font-weight", "bold"),
                                          ("font-size", "14px"), ("padding", "8px 0"),
                                          ("text-align", "left")]},
        {"selector": "th", "props": [("text-align", "left"), ("padding", "8px 12px"),
                                     ("border", "1px solid #cbd5e1")]},
    ])
)


## 7. Baseline reference and success conditions

The official baseline contains **31 false positives** and **22 false negatives**. The predefined numerical success conditions for a later intervention were:

- **FP < 31** — fewer legitimate emails incorrectly flagged as phishing.
- **FN ≤ 22** — no increase in the number of phishing emails missed.

The baseline provides a reference, **not an intervention result**. The ROC-AUC value cannot be reconstructed from a confusion matrix alone; it requires per-email prediction scores. Later explainability-guided analyses and corrected-model evaluations are documented separately.

## 8. Reproducibility and limitations

**Documented:** dataset and split counts, model identifier, recorded training configuration, official confusion matrix, and official metrics.

**Not performed in this notebook:** Kaggle download, text cleaning, tokenization, fine-tuning, checkpoint loading, probability inference, or ROC-AUC recalculation. The arithmetic checks confirm agreement among the published aggregate counts; they do not independently reproduce the original experiment.

The original official baseline serves as the reference for later **test-informed** SHAP-guided interventions. Those later evaluations must distinguish observed improvements from independently validated generalization.

## 9. Conclusion

The official DistilBERT baseline correctly classified **4,717 legitimate emails** and **5,946 phishing emails**. It produced **31 false positives** and **22 false negatives** on **10,716 test emails**, with recorded **accuracy of 0.995054**, **F1-score of 0.995563**, and **ROC-AUC of 0.999262**.

These locked baseline values are retained for subsequent comparisons. **No model was retrained and no evaluation result was changed in this notebook.**